# Feature Track 4e: RAG as a Subagent

The previous notebooks gave a single agent access to a retrieval tool. Here the full RAG agent from 4d is itself wrapped as a tool and handed to a **parent agent**.

This is the **subagent pattern**: the parent agent handles the conversation and routing; the RAG agent handles retrieval and grounding. Responsibilities are separated, the parent does not need to know about chunks or sources.

```
User ──► Main agent ──► (if domain question)  ──► RAG subagent ──► retriever ──► vector store
                 └────► (if general question) ──► answer directly
```

---

## Setup

**Prerequisites:** `conversational_toolkit` and `backend` installed in editable mode. Set `OPENAI_API_KEY`.

In [ ]:
from typing import Any
from conversational_toolkit.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.llms.base import LLMMessage, Roles, MessageContent
from conversational_toolkit.tools.base import Tool
from conversational_toolkit.llms.openai import OpenAILLM
from conversational_toolkit.vectorstores.chromadb import ChromaDBVectorStore
from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever
from conversational_toolkit.agents.tool_agent import ToolAgent, QueryWithContext

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_llm,
    build_vector_store,
    VS_PATH,
    EMBEDDING_MODEL,
)
from sme_kt_zh_collaboration_rag.feature4_tool_agents import RetrieveRelevantChunks

BACKEND = "openai"  # "ollama" or "openai"
FORCE_REBUILD = False  # set True to re-embed from scratch

if "sentence-transformers" in EMBEDDING_MODEL:
    embedding_model = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL)
else:
    embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)

if FORCE_REBUILD or not VS_PATH.exists():
    chunks = load_chunks(max_files=5)
    chunks = [c for c in chunks if c.mime_type.startswith("text")]
    db_chroma = await build_vector_store(
        chunks, embedding_model, db_path=VS_PATH, reset=FORCE_REBUILD
    )
else:
    # Vector store already exists -> open it without re-embedding
    db_chroma = ChromaDBVectorStore(db_path=str(VS_PATH))
    print(
        f"Reusing existing vector store at {VS_PATH} ({db_chroma.collection.count()} chunks)"
    )

vector_store = VectorStoreRetriever(embedding_model, db_chroma, top_k=5)

if not BACKEND:
    raise ValueError('Set BACKEND to "ollama" or "openai" before running.')

llm = build_llm(backend=BACKEND)

In [ ]:
retriever_tool = RetrieveRelevantChunks(
    name="retrieve_relevant_chunks",
    description="Retrieves the most relevant chunks based on a query.",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The query to retrieve relevant chunks for.",
            },
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    retriever=vector_store,
)

---

## Build the RAG Subagent

The RAG agent is identical to the one in 4d: a `ToolAgent` with a retrieval tool. It will be called by the parent agent, not directly by the user.

In [ ]:
llm = OpenAILLM(tool_choice="auto", tools=[retriever_tool])

prompt = "You are a helpful assistant, answer shortly. Use the tools only when they are relevant, but if you do so, trust the results from the tools and use them in your answer, cite them precisely if you use them."
prompt_as_message = LLMMessage(
    content=[MessageContent(type="text", text=prompt)], role=Roles.SYSTEM
)

rag_agent = ToolAgent(system_prompt=prompt, llm=llm, max_steps=5)

---

## Wrap the RAG Agent as a Tool

`RAGAgentAsTool` delegates its `call` to `rag_agent.answer`. From the parent agent's perspective, this is just another tool it can invoke with a query string, the internal retrieval loop is invisible.

In [ ]:
class RAGAgentAsTool(Tool):
    def __init__(
        self,
        name: str,
        description: str,
        parameters: dict[str, Any],
    ):
        self.name = name
        self.description = description
        self.parameters = parameters

    async def call(self, args: dict[str, Any]) -> dict[str, Any]:
        query = args.get("query")
        assert query is not None

        answer = await rag_agent.answer(
            query_with_context=QueryWithContext(query=query, history=[])
        )
        answer_as_text = str(answer.content)

        return {"result": answer_as_text}


rag_agent_as_tool_tool = RAGAgentAsTool(
    name="rag_agent_as_tool",
    description="Uses the RAG agent to answer queries about pellets, boxes and so on.",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The query to retrieve relevant chunks for.",
            },
        },
        "required": ["query"],
        "additionalProperties": False,
    },
)

---

## Define the Main Agent

The parent agent receives the RAG tool but has a simple, generic system prompt, it does not need to know anything about sources or retrieval. When the query requires domain knowledge, it delegates to the RAG subagent.

In [ ]:
llm_main_agent = OpenAILLM(tool_choice="auto", tools=[rag_agent_as_tool_tool])

prompt_main_agent = "You are a helpful assistant, answer shortly."
prompt_as_message_main_agent = LLMMessage(
    content=[MessageContent(type="text", text=prompt_main_agent)], role=Roles.SYSTEM
)

main_agent = ToolAgent(system_prompt=prompt_main_agent, llm=llm_main_agent, max_steps=5)

---

## Test the Agent

Same two queries as 4d, then a memory check. The general question is answered directly; the domain question triggers the RAG subagent call.

### First Simple Question

In [ ]:
conversation: list[LLMMessage] = [prompt_as_message_main_agent]

query = "What is a Einstein theory of relativity in the context of physics?"
query_as_message = LLMMessage(
    content=[MessageContent(type="text", text=query)], role=Roles.USER
)
query_with_context = QueryWithContext(query=query, history=[])

answer = await main_agent.answer(query_with_context)

print("\nAnswer:")
print(answer.content[0].text)

conversation = globals().get("conversation", [])
conversation += [query_as_message, answer]

### Domain Specific Question

In [ ]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_as_message = LLMMessage(
    content=[MessageContent(type="text", text=query)], role=Roles.USER
)
query_with_context = QueryWithContext(query=query, history=conversation)

answer = await main_agent.answer(query_with_context)

print("\nAnswer:")
print(answer.content[0].text)

conversation += [query_as_message, answer]

### Test Memory

In [ ]:
query = "Summarize the conversation (some words per message)?"
query_as_message = LLMMessage(
    content=[MessageContent(type="text", text=query)], role=Roles.USER
)
query_with_context = QueryWithContext(query=query, history=conversation)

answer = await main_agent.answer(query_with_context)

print("\nAnswer:")
print(answer.content[0].text)

----------------